In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.data.load_data import load_datasets

datasets = load_datasets()

Loaded 9 datasets successfully.


In [4]:
orders_df = datasets["olist_orders_dataset"].copy()

In [5]:
orders_df.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [7]:
import pandas as pd

In [8]:
for column in date_columns:

    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

In [9]:
orders_df.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Datetime Conversion

### Objective
Convert timestamp columns from `object` to `datetime` for time-based analysis.

### Findings
- Successfully converted all five timestamp columns.
- Invalid dates, if any, were converted to `NaT`.
- The dataset is now ready for time-series feature engineering.

In [10]:
from src.utils.data_profiler import (
    profile_summary,
    missing_value_report
)

In [11]:
missing_report = missing_value_report(orders_df)

display(missing_report)

,Column,Missing Values,Missing %,Data Type
6,order_delivered_customer_date,2965,2.98,datetime64[ns]
5,order_delivered_carrier_date,1783,1.79,datetime64[ns]
4,order_approved_at,160,0.16,datetime64[ns]
0,order_id,0,0.00,object
3,order_purchase_timestamp,0,0.00,datetime64[ns]
2,order_status,0,0.00,object
1,customer_id,0,0.00,object
7,order_estimated_delivery_date,0,0.00,datetime64[ns]


In [12]:
orders_df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders_df.loc[
    orders_df["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [14]:
orders_df.loc[
    orders_df["order_delivered_carrier_date"].isna(),
    "order_status"
].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [15]:
orders_df.loc[
    orders_df["order_approved_at"].isna(),
    "order_status"
].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [16]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## Missing Value Handling

### Objective
Investigate missing values and determine whether they represent data quality issues or valid business events.

### Decision
- Missing delivery timestamps were retained because they correspond to cancelled or unavailable orders.
- Missing approval timestamps were also retained for the same reason.
- No imputation was performed since filling these values would introduce incorrect business information.

### Outcome
The dataset preserves its business meaning while remaining suitable for downstream analysis.

In [17]:
duplicate_rows = orders_df.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")

Duplicate Rows: 0


In [18]:
orders_df["order_id"].nunique(), len(orders_df)

(99441, 99441)

In [19]:
order_items_df = datasets["olist_order_items_dataset"].copy()

In [20]:
order_items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [21]:
missing_value_report(order_items_df)

,Column,Missing Values,Missing %,Data Type
0,order_id,0,0.0,object
1,order_item_id,0,0.0,int64
2,product_id,0,0.0,object
3,seller_id,0,0.0,object
4,shipping_limit_date,0,0.0,object
5,price,0,0.0,float64
6,freight_value,0,0.0,float64


In [22]:
print("Duplicate Rows:", order_items_df.duplicated().sum())

Duplicate Rows: 0


In [23]:
duplicate_keys = order_items_df[
    ["order_id", "order_item_id"]
].duplicated().sum()

print("Duplicate Composite Keys:", duplicate_keys)

Duplicate Composite Keys: 0


In [24]:
order_items_df.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [25]:
order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

In [26]:
order_items_df.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

In [27]:
print(
    "Missing Shipping Dates:",
    order_items_df["shipping_limit_date"].isna().sum()
)

Missing Shipping Dates: 0


## Order Items Dataset Cleaning

### Objective
Clean and validate the order items dataset before joining it with other datasets.

### Findings
- No missing values found.
- No duplicate rows detected.
- Composite key (`order_id`, `order_item_id`) is valid.
- Converted `shipping_limit_date` from `object` to `datetime`.

### Outcome
The dataset is clean and ready for integration.

In [28]:
products_df = datasets["olist_products_dataset"].copy()

In [29]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [30]:
missing_value_report(products_df)

,Column,Missing Values,Missing %,Data Type
1,product_category_name,610,1.85,object
3,product_description_lenght,610,1.85,float64
2,product_name_lenght,610,1.85,float64
4,product_photos_qty,610,1.85,float64
5,product_weight_g,2,0.01,float64
7,product_height_cm,2,0.01,float64
6,product_length_cm,2,0.01,float64
8,product_width_cm,2,0.01,float64
0,product_id,0,0.00,object


In [31]:
print(
    "Duplicate Rows:",
    products_df.duplicated().sum()
)

Duplicate Rows: 0


In [32]:
print(
    "Unique Product IDs:",
    products_df["product_id"].nunique()
)

print(
    "Total Rows:",
    len(products_df)
)

Unique Product IDs: 32951
Total Rows: 32951


## Products Dataset Cleaning

### Objective
Validate the quality and completeness of the products dataset before integrating it with pricing and sales data.

### Checks Performed
- Verified data types.
- Analyzed missing values.
- Checked duplicate rows.
- Validated the primary key (`product_id`).

### Status
Further investigation is required for product attribute columns containing missing values.

In [33]:
products_df[
    products_df.isnull().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [34]:
products_df[
    products_df[
        [
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty"
        ]
    ].isnull().all(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [35]:
products_df = products_df.dropna(
    subset=[
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
)

In [36]:
missing_value_report(products_df)

,Column,Missing Values,Missing %,Data Type
8,product_width_cm,1,0.0,float64
6,product_length_cm,1,0.0,float64
7,product_height_cm,1,0.0,float64
5,product_weight_g,1,0.0,float64
0,product_id,0,0.0,object
4,product_photos_qty,0,0.0,float64
3,product_description_lenght,0,0.0,float64
2,product_name_lenght,0,0.0,float64
1,product_category_name,0,0.0,object


In [37]:
products_df[
    products_df[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN


## Products Dataset Cleaning Decisions

### Findings
- 610 products were missing multiple essential metadata fields.
- These represented approximately 1.85% of the dataset.
- Since the missing values occurred together and could not be reliably imputed, these products were removed.

### Remaining Investigation
- Two products still contain missing physical dimensions and weight.
- These rows will be investigated separately before making a final decision.

### Cleaning Decision

Two products were missing all physical dimension attributes
(weight, length, height, and width).

These attributes are essential for shipping and pricing analysis.

Since:

- only two products were affected,
- the missing values could not be reliably imputed,
- and removing them affects less than 0.01% of the dataset,

the rows were removed to maintain data quality.

In [38]:
customers_df = datasets["olist_customers_dataset"].copy()

In [39]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [40]:
missing_value_report(customers_df)

,Column,Missing Values,Missing %,Data Type
0,customer_id,0,0.0,object
1,customer_unique_id,0,0.0,object
2,customer_zip_code_prefix,0,0.0,int64
3,customer_city,0,0.0,object
4,customer_state,0,0.0,object


In [41]:
duplicate_rows = customers_df.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")

Duplicate Rows: 0


In [42]:
print(customers_df["customer_id"].nunique())

99441


In [43]:
customers_df["customer_unique_id"].nunique()

96096

In [44]:
customers_df["customer_id"].nunique()

99441

## Customers Dataset Cleaning

### Observation
The dataset contains customer identifiers and location information.

### Checks Performed
- Verified data types.
- Checked missing values.
- Checked duplicate rows.
- Validated the primary key.

### Result
The dataset is clean and ready for integration with the Orders dataset.

In [45]:
payments_df = datasets["olist_order_payments_dataset"].copy()

In [46]:
payments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [47]:
missing_value_report(payments_df)

,Column,Missing Values,Missing %,Data Type
0,order_id,0,0.0,object
1,payment_sequential,0,0.0,int64
2,payment_type,0,0.0,object
3,payment_installments,0,0.0,int64
4,payment_value,0,0.0,float64


In [48]:
print(
    "Duplicate Rows:",
    payments_df.duplicated().sum()
)

Duplicate Rows: 0


In [49]:
duplicate_keys = payments_df[
    ["order_id", "payment_sequential"]
].duplicated().sum()

print("Duplicate Composite Keys:", duplicate_keys)

Duplicate Composite Keys: 0


In [50]:
print(payments_df["payment_value"].describe())

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64


In [51]:
payments_df[
    payments_df["payment_value"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


## Payments Dataset Cleaning

### Observation
The payments dataset records transaction details for each order.

### Checks Performed
- Verified data types.
- Checked missing values.
- Checked duplicate rows.
- Validated the composite key (`order_id`, `payment_sequential`).
- Verified that payment values are valid.

### Result
The payments dataset is clean and ready for integration.

In [52]:
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(exist_ok=True)

In [53]:
orders_df.to_csv(
    PROCESSED_PATH / "orders_clean.csv",
    index=False
)

order_items_df.to_csv(
    PROCESSED_PATH / "order_items_clean.csv",
    index=False
)

products_df.to_csv(
    PROCESSED_PATH / "products_clean.csv",
    index=False
)

customers_df.to_csv(
    PROCESSED_PATH / "customers_clean.csv",
    index=False
)

payments_df.to_csv(
    PROCESSED_PATH / "payments_clean.csv",
    index=False
)